# CIS 5370 Final Project

## Project 1 - Intrusion Detection for Industrial Control Systems

- Kyle Brindle
- Rayhan Alcena

In [77]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

## Loading Files

In [78]:
# Base path to the datasets
BASE_PATH = Path.cwd() / "data/SWaT_A6_Dec_2019/"
SENSOR_READINGS_PATH = BASE_PATH / "csv/"
NETWORK_READINGS_PATH = BASE_PATH / "pcap/"

# Sensor Readings CSV/XLSX file paths
dec_2019_file_path = SENSOR_READINGS_PATH / "Dec2019.xlsx"

# Network Readings CSV/XLSX file paths
test_converted_pcap_file_path = NETWORK_READINGS_PATH / "pcap_00_converted.csv"

# Loading the CSV/XLSX files (We had to skip the first 9 rows (0-9))
sensor_readings_df = pd.read_excel(dec_2019_file_path, skiprows=9)

In [79]:
# Shows the first 10 observations in the dataframe
sensor_readings_df.head(10)

,t_stamp,P1_STATE,LIT101.Pv,FIT101.Pv,MV101.Status,P101.Status,P102.Status,P2_STATE,FIT201.Pv,AIT201.Pv,...,FIT601.Pv,P601.Status,P602.Status,P603.Status,LSH601.Alarm,LSL601.Alarm,LSH602.Alarm,LSL602.Alarm,LSH603.Alarm,LSL603.Alarm
0,2019-12-06 10:05:00,3,658.661255,0.0,1,2.0,1,2,2.313523,35.215330,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
1,2019-12-06 10:05:01,3,659.171600,0.0,1,2.0,1,2,2.311857,35.215330,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
2,2019-12-06 10:05:02,3,659.681800,0.0,1,2.0,1,2,2.311601,35.215330,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
3,2019-12-06 10:05:03,3,660.349100,0.0,1,2.0,1,2,2.310448,35.215330,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
4,2019-12-06 10:05:04,3,660.780945,0.0,1,2.0,1,2,2.310448,35.215330,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
5,2019-12-06 10:05:05,3,661.016400,0.0,1,2.0,1,2,2.311088,35.087160,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
6,2019-12-06 10:05:06,3,661.055700,0.0,1,2.0,1,2,2.316086,34.958984,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
7,2019-12-06 10:05:07,3,661.330444,0.0,1,2.0,1,2,2.317624,34.958984,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
8,2019-12-06 10:05:08,3,661.369700,0.0,1,2.0,1,2,2.321469,34.958984,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active
9,2019-12-06 10:05:09,3,661.644500,0.0,1,2.0,1,2,2.325057,34.958984,...,0.000256,2,1,1,Inactive,Inactive,Active,Inactive,Inactive,Active


## Preprocessing

In [80]:
# Separating the timestamps
t_stamp_sensor_readings_df = sensor_readings_df["t_stamp"]
print(t_stamp_sensor_readings_df)

0       2019-12-06 10:05:00
1       2019-12-06 10:05:01
2       2019-12-06 10:05:02
3       2019-12-06 10:05:03
4       2019-12-06 10:05:04
                ...        
13196   2019-12-06 13:44:56
13197   2019-12-06 13:44:57
13198   2019-12-06 13:44:58
13199   2019-12-06 13:44:59
13200   2019-12-06 13:45:00
Name: t_stamp, Length: 13201, dtype: datetime64[us]


In [81]:
# Identifying alarm columns for encoding
alarm_cols = [col for col in sensor_readings_df.columns if "Alarm" in col]
print("Alarm columns:", alarm_cols)

# Encoding alarm columns where Active = 1, Inactive = 0
for col in alarm_cols:
    sensor_readings_df[col] = (sensor_readings_df[col] == "Active").astype(int)

Alarm columns: ['LS201.Alarm', 'LS202.Alarm', 'LSL203.Alarm', 'LSLL203.Alarm', 'PSH301.Alarm', 'DPSH301.Alarm', 'LS401.Alarm', 'PSH501.Alarm', 'PSL501.Alarm', 'LSH601.Alarm', 'LSL601.Alarm', 'LSH602.Alarm', 'LSL602.Alarm', 'LSH603.Alarm', 'LSL603.Alarm']


In [82]:
# Creating a label column where Normal = 0, Attack = 1
sensor_readings_df["label"] = 0

# Attack windows for sensor/actuator disruption
# Format: YYYY-MM-DD HH:MM:SS
attack_windows = [
    ("2019-12-06 12:30:00", "2019-12-06 12:33:00"),
    ("2019-12-06 12:43:00", "2019-12-06 12:46:00"),
    ("2019-12-06 12:56:00", "2019-12-06 12:59:00"),
    ("2019-12-06 13:09:00", "2019-12-06 13:12:00"),
    ("2019-12-06 13:22:00", "2019-12-06 13:25:00"),
]

# Setting the label column for each observation as Normal or an Attack
# Based on the timestamp
for start, end in attack_windows:
    mask = (sensor_readings_df["t_stamp"] >= start) & (sensor_readings_df["t_stamp"] <= end)
    sensor_readings_df.loc[mask, "label"] = 1

# Verifying the label column
print(sensor_readings_df["label"].value_counts())
print(f"Attack: {sensor_readings_df["label"].mean() * 100}%")

label
0    12296
1      905
Name: count, dtype: int64
Attack: 6.855541246875236%


## Sensor Readings Evaluations

### Train/Test Split Based on the Timeline

In [83]:
# Setting up the training data with Normal sensor operations before the attack
train_sensor_readings_df = sensor_readings_df[t_stamp_sensor_readings_df < "2019-12-06 12:30:00"].drop(columns=["t_stamp", "label"])

# Setting up the testing data with the Full sensor operations, includes both Normal and Attack labels
test_sensor_readings_df = sensor_readings_df.drop(columns="t_stamp")
test_sensor_readings_labels = test_sensor_readings_df.pop("label")

print("Train shape:", train_sensor_readings_df.shape)
print("Test shape:", test_sensor_readings_df.shape)
print("Attack samples in test:", test_sensor_readings_labels.sum())

Train shape: (8700, 81)
Test shape: (13201, 81)
Attack samples in test: 905


### Normalization and Modeling

In [84]:
# Initializing the MinMaxScaler
scaler = MinMaxScaler()

# Fitting the training data
X_train_sensor_readings_df = scaler.fit_transform(X_train_sensor_readings_df)

# Testing
X_test_sensor_readings_df = scaler.transform(X_test_sensor_readings_df)

print("X_train shape:", X_train_sensor_readings_df.shape)
print("X_test shape:", X_test_sensor_readings_df.shape)

X_train shape: (8700, 33)
X_test shape: (13201, 33)


#### Model 1 - Statistical Threshold

In [86]:
# Computing the mean and standard deviation from the training data
mean_train_sensor_readings =  X_train_sensor_readings_df.mean(axis=0)
std_train_sensor_readings =  X_train_sensor_readings_df.std(axis=0)

# Flag  anamoly if any feature exceeds the mean plus/minus 3 times the standard deviation
z_scores = np.abs((X_test_sensor_readings_df - mean_train_sensor_readings) / (std_train_sensor_readings + 1e-8))

# Max z-score per row
anomaly_scores = z_scores.max(axis=1)

# Replace NaN in anomaly scores with 0
anomaly_scores = np.nan_to_num(anomaly_scores, nan=0.0)

# Predictions
threshold_predictions = (anomaly_scores > 3).astype(int)

print("=== Statistical Threshold ===")
print(classification_report(Y_test_sensor_readings_labels, threshold_predictions))
print("AUC:", roc_auc_score(Y_test_sensor_readings_labels, anomaly_scores))

=== Statistical Threshold ===
              precision    recall  f1-score   support

           0       0.96      0.67      0.79     12296
           1       0.12      0.60      0.20       905

    accuracy                           0.67     13201
   macro avg       0.54      0.63      0.49     13201
weighted avg       0.90      0.67      0.75     13201

AUC: 0.7144970111108315


## Network Readings Evaluations